# 05.2 Capstone: Text Classification

This notebook builds an offline-runnable text classification capstone project. The dataset is synthetic so the project remains stable in the current environment. The focus is the workflow: generate data, build baselines, train a sequence model, compare results, and explain what the comparison means.

The important modeling idea is that word order can matter. A unigram model sees individual words, while a bigram model and an LSTM have more direct ways to represent local order or sequence structure.

## Learning Goals

After this notebook, you should be able to:

1. Run a complete text classification project.
2. Compare a `bag-of-words` baseline with an `LSTM`.
3. Understand the project-level use of tokenization, vocabulary, and padding.
4. Organize train, validation, and test workflows.
5. Analyze order-sensitive phenomena such as negation.
6. Turn experiment results into a clear summary.

In [ ]:
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

## Generate an Order-Sensitive Dataset

The dataset is intentionally designed to include negation patterns. For example, `not bad movie` is positive, while `not good movie` is negative. These examples force the model to care about word combinations, not just the presence of a single word.

This makes the comparison between unigram, bigram, and LSTM models meaningful: each model has a different ability to represent word order.

In [ ]:
nouns = ["movie", "film", "story", "show", "plot", "episode"]
positive_templates = [
    ["good", "{noun}"],
    ["really", "good", "{noun}"],
    ["very", "fun", "{noun}"],
    ["not", "bad", "{noun}"],
    ["not", "boring", "{noun}"],
    ["quite", "nice", "{noun}"],
    ["love", "this", "{noun}"],
    ["enjoyable", "{noun}"],
]
negative_templates = [
    ["bad", "{noun}"],
    ["really", "bad", "{noun}"],
    ["very", "boring", "{noun}"],
    ["not", "good", "{noun}"],
    ["not", "fun", "{noun}"],
    ["quite", "weak", "{noun}"],
    ["hate", "this", "{noun}"],
    ["awful", "{noun}"],
]
optional_prefixes = [[], ["overall"], ["today"], ["honestly"], ["for", "me"]]
optional_suffixes = [[], ["overall"], ["for", "me"], ["today"]]


def render_template(template, noun):
    return [token.format(noun=noun) for token in template]


def make_dataset(n_per_label=700):
    texts = []
    labels = []

    for _ in range(n_per_label):
        noun = random.choice(nouns)
        prefix = random.choice(optional_prefixes)
        suffix = random.choice(optional_suffixes)
        tokens = prefix + render_template(random.choice(positive_templates), noun) + suffix
        texts.append(" ".join(tokens))
        labels.append(1)

    for _ in range(n_per_label):
        noun = random.choice(nouns)
        prefix = random.choice(optional_prefixes)
        suffix = random.choice(optional_suffixes)
        tokens = prefix + render_template(random.choice(negative_templates), noun) + suffix
        texts.append(" ".join(tokens))
        labels.append(0)

    combined = list(zip(texts, labels))
    random.shuffle(combined)
    texts, labels = zip(*combined)
    return list(texts), list(labels)


texts, labels = make_dataset(n_per_label=700)
print("dataset size =", len(texts))
print("positive rate / positive rate =", np.mean(labels))
print("sample texts =")
for text, label in list(zip(texts, labels))[:6]:
    print(label, "|", text)

## Train / Validation / Test Split

An important project habit is: do not only do train/test.

The validation set is used to compare ideas; the test set is used for final reporting.


In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

print("train size =", len(X_train))
print("val size =", len(X_val))
print("test size =", len(X_test))

## 3. Baseline 1: Unigram Bag-of-Words

The first baseline uses `CountVectorizer(ngram_range=(1, 1))` with `LogisticRegression`. It counts individual words and ignores word order. This is simple and fast, but it cannot directly distinguish `good movie` from `not good movie` as a phrase.

That weakness is intentional here, because it gives us a baseline that should struggle with negation.

In [ ]:
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
X_train_uni = unigram_vectorizer.fit_transform(X_train)
X_val_uni = unigram_vectorizer.transform(X_val)
X_test_uni = unigram_vectorizer.transform(X_test)

unigram_lr = LogisticRegression(max_iter=2000, random_state=42)
unigram_lr.fit(X_train_uni, y_train)

unigram_val_preds = unigram_lr.predict(X_val_uni)
unigram_test_preds = unigram_lr.predict(X_test_uni)

unigram_val_acc = accuracy_score(y_val, unigram_val_preds)
unigram_test_acc = accuracy_score(y_test, unigram_test_preds)

print("unigram val acc =", round(unigram_val_acc, 4))
print("unigram test acc =", round(unigram_test_acc, 4))

## 4. Baseline 2: Bigram Bag-of-Words

The second baseline adds `bigrams`.

This can usually handle local order patterns such as `not good` better.


In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_train_bi = bigram_vectorizer.fit_transform(X_train)
X_val_bi = bigram_vectorizer.transform(X_val)
X_test_bi = bigram_vectorizer.transform(X_test)

bigram_lr = LogisticRegression(max_iter=2000, random_state=42)
bigram_lr.fit(X_train_bi, y_train)

bigram_val_preds = bigram_lr.predict(X_val_bi)
bigram_test_preds = bigram_lr.predict(X_test_bi)

bigram_val_acc = accuracy_score(y_val, bigram_val_preds)
bigram_test_acc = accuracy_score(y_test, bigram_test_preds)

print("bigram val acc =", round(bigram_val_acc, 4))
print("bigram test acc =", round(bigram_test_acc, 4))

## Prepare Data for the LSTM

Now we switch to the `PyTorch` sequence-modeling route.

We need three steps here:

1. `tokenization
2. `vocabulary
3. `padding

In [ ]:
def tokenize(text):
    return text.split()


special_tokens = ["<pad>", "<unk>"]
vocab = sorted({token for text in X_train for token in tokenize(text)})
vocab = special_tokens + vocab
stoi = {token: idx for idx, token in enumerate(vocab)}
pad_id = stoi["<pad>"]
unk_id = stoi["<unk>"]
max_len = max(len(tokenize(text)) for text in X_train)

print("vocab size =", len(vocab))
print("max_len =", max_len)
print("first vocab items =", vocab[:12])

In [ ]:
def encode_text(text, max_len=max_len):
    tokens = tokenize(text)
    ids = [stoi.get(token, unk_id) for token in tokens][:max_len]
    while len(ids) < max_len:
        ids.append(pad_id)
    return ids


def build_tensor_dataset(texts, labels):
    x = torch.tensor([encode_text(text) for text in texts], dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.long)
    return TensorDataset(x, y)


train_ds = build_tensor_dataset(X_train, y_train)
val_ds = build_tensor_dataset(X_val, y_val)
test_ds = build_tensor_dataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)
print("sample encoded =", xb[0])

## LSTM Model and Training Function

An `LSTM` reads tokens in sequence order.

So in principle it is better suited than unigram bag-of-words for order-sensitive patterns.


In [ ]:
class TextLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=48, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        emb = self.embedding(x)
        _, (h_n, _) = self.lstm(emb)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * xb.size(0)
            total_correct += (preds == yb).sum().item()
            total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


def train_lstm(config):
    set_seed(42)
    model = TextLSTMClassifier(
        vocab_size=len(vocab),
        embed_dim=config["embed_dim"],
        hidden_dim=config["hidden_dim"],
        pad_id=pad_id,
    )
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

    history = []
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0

    for epoch in range(1, config["epochs"] + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def collect_predictions(model, loader):
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb).argmax(dim=1))
            targets.append(yb)
    return torch.cat(preds), torch.cat(targets)

## Train the LSTM

Here we use the `LSTM` as the sequence-model route.


In [ ]:
lstm_config = {
    "embed_dim": 32,
    "hidden_dim": 48,
    "lr": 0.01,
    "epochs": 10,
}

lstm_model, lstm_history = train_lstm(lstm_config)
lstm_val_preds, lstm_val_targets = collect_predictions(lstm_model, val_loader)
lstm_test_preds, lstm_test_targets = collect_predictions(lstm_model, test_loader)

lstm_val_acc = accuracy_score(lstm_val_targets.numpy(), lstm_val_preds.numpy())
lstm_test_acc = accuracy_score(lstm_test_targets.numpy(), lstm_test_preds.numpy())

print("lstm val acc =", round(lstm_val_acc, 4))
print("lstm test acc =", round(lstm_test_acc, 4))

## Result Table

The result table compares unigram bag-of-words, bigram bag-of-words, and LSTM models under the same dataset split. The table is not just for ranking models; it helps explain what kind of structure the task requires.

If bigrams close most of the gap, then local two-word phrases are probably enough for this synthetic task. If the LSTM wins by a lot, then longer sequence structure may matter more.

In [ ]:
results_df = pd.DataFrame(
    [
        {
            "model": "Unigram-LR",
            "family": "baseline",
            "val_acc": round(unigram_val_acc, 4),
            "test_acc": round(unigram_test_acc, 4),
            "notes": "bag-of-words without order",
        },
        {
            "model": "Bigram-LR",
            "family": "baseline",
            "val_acc": round(bigram_val_acc, 4),
            "test_acc": round(bigram_test_acc, 4),
            "notes": "bag-of-words with local order patterns",
        },
        {
            "model": "LSTM",
            "family": "torch",
            "val_acc": round(lstm_val_acc, 4),
            "test_acc": round(lstm_test_acc, 4),
            "notes": "sequence model with recurrent order modeling",
        },
    ]
).sort_values(by=["test_acc", "val_acc"], ascending=False)
results_df

## LSTM Training Curves

We only plot the `LSTM` training curves because the two `LogisticRegression` baselines do not have epoch histories.


In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(lstm_history["epoch"], lstm_history["train_loss"], label="train loss")
plt.plot(lstm_history["epoch"], lstm_history["val_loss"], label="val loss")
plt.title("LSTM Loss Curves")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(lstm_history["epoch"], lstm_history["train_acc"], label="train acc")
plt.plot(lstm_history["epoch"], lstm_history["val_acc"], label="val acc")
plt.title("LSTM Accuracy Curves")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.tight_layout()
plt.show()
plt.close()

## Select the Best Model

Here we again select the final model by the highest `test_acc`, only for teaching demonstration.


In [ ]:
all_candidates = {
    "Unigram-LR": {
        "preds": np.array(unigram_test_preds),
        "targets": np.array(y_test),
        "test_acc": unigram_test_acc,
    },
    "Bigram-LR": {
        "preds": np.array(bigram_test_preds),
        "targets": np.array(y_test),
        "test_acc": bigram_test_acc,
    },
    "LSTM": {
        "preds": lstm_test_preds.numpy(),
        "targets": lstm_test_targets.numpy(),
        "test_acc": lstm_test_acc,
    },
}

best_name = max(all_candidates, key=lambda name: all_candidates[name]["test_acc"])
best_preds = all_candidates[best_name]["preds"]
best_targets = all_candidates[best_name]["targets"]

print("best model =", best_name)
print("best test acc =", round(all_candidates[best_name]["test_acc"], 4))

## 11. Confusion Matrix

This is binary classification, so the matrix is small, but it is still useful.

It helps you see whether the model is more likely to miss positives or miss negatives.


In [ ]:
cm = confusion_matrix(best_targets, best_preds)
plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap="Blues")
plt.title(f"Confusion Matrix: {best_name}")
plt.xlabel("predicted label")
plt.ylabel("true label")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
plt.show()
plt.close()

print(classification_report(best_targets, best_preds, digits=4, target_names=["negative", "positive"]))

## Misclassified Text Analysis

In text tasks, the misclassified examples are often more interpretable than the metric itself.


In [ ]:
mis_positions = np.where(best_preds != best_targets)[0]
mis_rows = []
for pos in mis_positions[:12]:
    mis_rows.append(
        {
            "text": X_test[pos],
            "true_label": int(best_targets[pos]),
            "pred_label": int(best_preds[pos]),
        }
    )

mis_df = pd.DataFrame(mis_rows)
print("num misclassified =", len(mis_positions))
mis_df

## Interpreting the Results

One key question in this project is:

- how much modeling power is needed for local order patterns such as `not good`?

If `Bigram-LR` is already very strong, it means much of the information can be captured by local n-grams.

If the `LSTM` further improves performance, it suggests that fuller sequence modeling is indeed helpful.


In [ ]:
summary_lines = [
    f"Best model: {best_name}",
    f"Unigram-LR test accuracy: {unigram_test_acc:.4f}",
    f"Bigram-LR test accuracy: {bigram_test_acc:.4f}",
    f"LSTM test accuracy: {lstm_test_acc:.4f}",
    f"Number of misclassified test samples: {len(mis_positions)}",
]

for line in summary_lines:
    print(line)

In [ ]:
# Exercise 1
#
# Explain why a unigram bag-of-words model can struggle with "not good" versus
# "good".
#
# Answer in full sentences. Your answer should mention that a unigram model sees
# individual word counts but does not directly encode the local phrase "not good".

Exercise 1 Reference Answer

Because a unigram model only looks at individual words and does not directly represent the order relation between adjacent words.

It knows that `not` appears and that `good` appears, but not necessarily that they form the phrase `not good`.


In [ ]:
# Exercise 2
#
# If Bigram-LR is already very close to the LSTM, what does that suggest about
# the task?
#
# Answer in full sentences. Your answer should mention whether local phrase
# patterns are sufficient and why a more complex sequence model may not add much
# on this dataset.

Exercise 2 Reference Answer

This usually suggests that in the current task, most of the crucial information can be captured by local phrase patterns.

In other words, a sequence model does not always win by a large margin; whether the extra complexity is worth it depends on the task.


## Summary

In this text capstone, you have practiced the full chain:

1. task construction
2. text preprocessing
3. baseline comparison
4. `LSTM` sequence modeling
5. result tables, training curves, confusion matrices, and misclassification analysis
6. experiment conclusion writing